# AI Video Tracking - Colab Backend

This notebook runs the AI Video Tracking backend on Google Colab to leverage GPU acceleration.

### 1. Enable GPU
Go to **Edit** -> **Notebook settings** -> Select **T4 GPU** (or any available GPU).

### 2. Upload `backend.zip`
Upload the `backend.zip` file created by `collab/prepare_backend.py` to the current Colab directory.

In [ ]:
# @title Install Dependencies
!pip install -U ultralytics supervision fastapi uvicorn pyngrok python-multipart imageio[ffmpeg] pandas numpy opencv-python nest_asyncio

In [ ]:
# @title Download Advanced Model
import os
from ultralytics import YOLO
import shutil

# Pre-download YOLOv12m for better tracking precision
model_name = 'yolo12m.pt'
print(f'Downloading {model_name}...')
model = YOLO(model_name)

# Ensure models directory exists in the app folder
os.makedirs('backend_app/models', exist_ok=True)

# Move to the backend folder if it exists
if os.path.exists('backend_app/models'):
    shutil.move(model_name, f'backend_app/models/{model_name}')
    print(f'Model moved to backend_app/models/{model_name}')

In [ ]:
# @title Extract Backend
import os
import zipfile

if os.path.exists('backend.zip'):
    with zipfile.ZipFile('backend.zip', 'r') as zip_ref:
        zip_ref.extractall('backend_app')
    print("Backend extracted to 'backend_app'")
else:
    print("Error: backend.zip not found. Please upload it.")

In [ ]:
# @title Run Backend with Ngrok
import getpass
import os
import time
import nest_asyncio
from pyngrok import ngrok
import uvicorn

# Apply nest_asyncio to allow uvicorn to run in the existing Colab event loop
nest_asyncio.apply()

# 1. Aggressively kill any local ngrok processes
!pkill ngrok
time.sleep(2)

try:
    ngrok.kill()
    print("Cleaned up local ngrok processes.")
except:
    pass

print("NOTE: If you still see ERR_NGROK_334, go to https://dashboard.ngrok.com/tunnels/agents and stop any active tunnels manually.")

# Enter your ngrok authtoken here
print("Get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken")
authtoken = getpass.getpass("Enter ngrok authtoken: ")
ngrok.set_auth_token(authtoken)

# Open an ngrok tunnel to the HTTP server
public_url = ngrok.connect(8000).public_url
print(f"\n * PUBLIC API URL: {public_url} *")
print("Update your frontend/src/services/api.ts with this URL.\n")

# Change directory to the extracted backend
if os.path.exists('/content/backend_app'):
    os.chdir('/content/backend_app')
else:
    # Fallback for different environments
    if os.path.exists('backend_app'):
        os.chdir('backend_app')

# Run the app using uvicorn
if __name__ == "__main__":
    config = uvicorn.Config("api.app:app", host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    await server.serve()